# Notebook 11 — 3D Representations & Point Cloud Processing

**Vision & 3D Mapping Workshop** | Block 3: 3D Mapping

---

## Why This Matters

Autonomous systems perceive the world through sensors that produce *depth measurements* — stereo
cameras, LiDAR, structured-light sensors, time-of-flight cameras. The raw data is a 2-D grid of
distances, but the downstream tasks — obstacle avoidance, manipulation, mapping — require full
3-D understanding.  Converting depth images into point clouds, cleaning and aligning those clouds,
and choosing the right 3-D representation are foundational skills for any 3-D vision pipeline.

This notebook derives every algorithm **from first principles**, implements it from scratch in
NumPy, and then compares it to our workshop library (`src/pointcloud.py`).

### What You'll Learn

1. **Back-projection** — inverting the pinhole model to lift pixels into 3-D
2. **Point cloud operations** — voxel downsampling, normal estimation, outlier removal
3. **ICP alignment** — point-to-point and point-to-plane registration
4. **Poisson surface reconstruction** — from oriented points to watertight meshes
5. **Representation comparison** — point clouds vs. voxels vs. meshes vs. neural implicits
6. **SE(3)-equivariant networks** — architectures that respect geometric symmetries
7. **Exercises** — hands-on practice with all of the above

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.spatial import KDTree

np.set_printoptions(precision=6, suppress=True)
%matplotlib inline

import src.pointcloud as pc
from src.camera import CameraIntrinsics

---

## 1. Back-Projection: Depth Map → Point Cloud

### 1.1 The Pinhole Camera Model (Review)

The **pinhole projection** maps a 3-D point $\mathbf{P} = (X, Y, Z)^T$ in the camera frame
to a pixel $(u, v)$:

$$
\lambda \begin{pmatrix} u \\ v \\ 1 \end{pmatrix}
= K \begin{pmatrix} X \\ Y \\ Z \end{pmatrix}
= \begin{pmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{pmatrix}
  \begin{pmatrix} X \\ Y \\ Z \end{pmatrix}
$$

where $\lambda = Z$ is the projective depth, $(f_x, f_y)$ are focal lengths in pixels,
and $(c_x, c_y)$ is the principal point.

Expanding:

$$
u = f_x \frac{X}{Z} + c_x, \qquad v = f_y \frac{Y}{Z} + c_y.
$$

### 1.2 Inverting the Pinhole Model

**Back-projection** is the inverse operation: given a pixel $(u, v)$ and its depth
$d = Z$, recover the 3-D point.  Solving the projection equations for $(X, Y, Z)$:

$$
\boxed{
X = \frac{(u - c_x) \cdot d}{f_x}, \qquad
Y = \frac{(v - c_y) \cdot d}{f_y}, \qquad
Z = d.
}
$$

**Derivation.** Start from the projection equation $u = f_x X/Z + c_x$.  Substituting $Z = d$:

$$
u = f_x \frac{X}{d} + c_x \;\Longrightarrow\; u - c_x = f_x \frac{X}{d}
\;\Longrightarrow\; X = \frac{(u - c_x) \cdot d}{f_x}.
$$

The $Y$ equation follows identically.  In matrix form:

$$
\begin{pmatrix} X \\ Y \\ Z \end{pmatrix}
= d \cdot K^{-1} \begin{pmatrix} u \\ v \\ 1 \end{pmatrix}
= d \begin{pmatrix} 1/f_x & 0 & -c_x/f_x \\ 0 & 1/f_y & -c_y/f_y \\ 0 & 0 & 1 \end{pmatrix}
  \begin{pmatrix} u \\ v \\ 1 \end{pmatrix}.
$$

This is the fundamental equation behind *every* depth-camera-based 3-D reconstruction pipeline.

### 1.3 Vectorised Implementation

For a depth image of size $H \times W$, we have up to $H \cdot W$ pixels to back-project.
A naive double loop in Python would be prohibitively slow.  Instead we:

1. Build coordinate grids $\mathbf{U}, \mathbf{V} \in \mathbb{R}^{H \times W}$ via `np.meshgrid`.
2. Apply the back-projection formula element-wise on entire arrays.
3. Mask out invalid pixels ($d = 0$).
4. Stack the resulting $(X, Y, Z)$ arrays into an $(N, 3)$ point cloud.

In [ ]:
def depth_to_pointcloud_scratch(depth, fx, fy, cx, cy):
    """Back-project a depth map to a 3-D point cloud (from scratch)."""
    H, W = depth.shape
    u_coords, v_coords = np.meshgrid(np.arange(W, dtype=np.float64),
                                     np.arange(H, dtype=np.float64))

    valid = depth > 0
    u = u_coords[valid]
    v = v_coords[valid]
    d = depth[valid].astype(np.float64)

    X = (u - cx) * d / fx
    Y = (v - cy) * d / fy
    Z = d

    return np.stack([X, Y, Z], axis=-1)

### 1.4 Synthetic Depth Map & Visualisation

We generate a synthetic depth map containing a **tilted plane** and a **sphere**.  This lets
us verify back-projection visually without needing real sensor data.

In [ ]:
H, W = 120, 160
fx, fy = 200.0, 200.0
cx, cy = W / 2.0, H / 2.0

K = np.array([[fx, 0, cx],
              [0, fy, cy],
              [0,  0,  1]])

u_grid, v_grid = np.meshgrid(np.arange(W), np.arange(H))

depth_plane = 3.0 + 0.005 * (u_grid - W/2) + 0.008 * (v_grid - H/2)

sphere_cx_px, sphere_cy_px = W // 2, H // 2
sphere_radius_px = 25
sphere_depth_centre = 2.5
dist_from_centre = np.sqrt((u_grid - sphere_cx_px)**2 + (v_grid - sphere_cy_px)**2)
sphere_mask = dist_from_centre < sphere_radius_px
sphere_z = np.sqrt(np.maximum(sphere_radius_px**2 - dist_from_centre**2, 0)) / fx
depth_map = depth_plane.copy()
depth_map[sphere_mask] = sphere_depth_centre - sphere_z[sphere_mask]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
im = axes[0].imshow(depth_map, cmap='viridis')
axes[0].set_title('Synthetic Depth Map')
plt.colorbar(im, ax=axes[0], label='Depth (m)')

pts = depth_to_pointcloud_scratch(depth_map, fx, fy, cx, cy)
print(f"Point cloud shape: {pts.shape}")

ax3d = fig.add_subplot(122, projection='3d')
subsample = np.random.choice(len(pts), min(4000, len(pts)), replace=False)
ax3d.scatter(pts[subsample, 0], pts[subsample, 1], pts[subsample, 2],
             c=pts[subsample, 2], cmap='viridis', s=1, alpha=0.6)
ax3d.set_xlabel('X'); ax3d.set_ylabel('Y'); ax3d.set_zlabel('Z')
ax3d.set_title('Back-Projected Point Cloud')
ax3d.view_init(elev=25, azim=-60)
plt.tight_layout()
plt.show()

---

## 2. Point Cloud Operations

Raw point clouds from depth sensors are typically noisy, unevenly sampled, and contain
millions of points.  Before any downstream processing we need to:

1. **Downsample** to a manageable size while preserving geometry.
2. **Estimate surface normals** for rendering and registration.
3. **Remove outliers** caused by sensor noise, multi-path reflections, or edge artefacts.

---

### 2.1 Voxel Downsampling

#### Algorithm

Voxel downsampling partitions 3-D space into a regular grid of cubes (voxels) with side
length $s$, then replaces all points in each occupied voxel with their **centroid**.

For a point $\mathbf{p} = (x, y, z)^T$, its voxel index is:

$$
\mathbf{v} = \left\lfloor \frac{\mathbf{p}}{s} \right\rfloor
= \left( \left\lfloor \frac{x}{s} \right\rfloor,\;
   \left\lfloor \frac{y}{s} \right\rfloor,\;
   \left\lfloor \frac{z}{s} \right\rfloor \right)^T.
$$

Given a set of $N$ points, the algorithm:

1. Compute voxel indices $\mathbf{v}_i$ for all points.
2. Hash each 3-D index to a scalar key: $k = v_x \cdot D_y D_z + v_y \cdot D_z + v_z$.
3. Group points by key.
4. Output the centroid $\bar{\mathbf{p}}_j = \frac{1}{|\mathcal{V}_j|} \sum_{i \in \mathcal{V}_j} \mathbf{p}_i$ for each occupied voxel $\mathcal{V}_j$.

**Complexity**: $O(N \log N)$ due to sorting (or $O(N)$ with hash maps).

#### From-Scratch Implementation

In [ ]:
def voxel_downsample_scratch(points, voxel_size):
    """Downsample a point cloud using a voxel grid filter (from scratch)."""
    voxel_idx = np.floor(points[:, :3] / voxel_size).astype(np.int64)

    mins = voxel_idx.min(axis=0)
    shifted = voxel_idx - mins
    dims = shifted.max(axis=0) + 1
    keys = (shifted[:, 0] * (dims[1] * dims[2])
            + shifted[:, 1] * dims[2]
            + shifted[:, 2])

    order = np.argsort(keys)
    sorted_keys = keys[order]
    sorted_pts = points[order]

    splits = np.flatnonzero(np.diff(sorted_keys)) + 1
    groups = np.split(sorted_pts, splits)

    centroids = np.array([g.mean(axis=0) for g in groups])
    return centroids

In [ ]:
np.random.seed(42)
cloud_raw = pts + np.random.normal(0, 0.005, pts.shape)

voxel_size = 0.05
cloud_down = voxel_downsample_scratch(cloud_raw, voxel_size)

print(f"Original: {cloud_raw.shape[0]:,} points")
print(f"After voxel downsample (s={voxel_size}): {cloud_down.shape[0]:,} points")
print(f"Reduction ratio: {cloud_down.shape[0] / cloud_raw.shape[0]:.2%}")

fig = plt.figure(figsize=(14, 5))
ax1 = fig.add_subplot(121, projection='3d')
sub = np.random.choice(len(cloud_raw), min(4000, len(cloud_raw)), replace=False)
ax1.scatter(cloud_raw[sub, 0], cloud_raw[sub, 1], cloud_raw[sub, 2],
            c=cloud_raw[sub, 2], cmap='viridis', s=1, alpha=0.5)
ax1.set_title(f'Original ({cloud_raw.shape[0]:,} pts)')
ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')
ax1.view_init(elev=25, azim=-60)

ax2 = fig.add_subplot(122, projection='3d')
ax2.scatter(cloud_down[:, 0], cloud_down[:, 1], cloud_down[:, 2],
            c=cloud_down[:, 2], cmap='viridis', s=3, alpha=0.7)
ax2.set_title(f'Downsampled ({cloud_down.shape[0]:,} pts, s={voxel_size})')
ax2.set_xlabel('X'); ax2.set_ylabel('Y'); ax2.set_zlabel('Z')
ax2.view_init(elev=25, azim=-60)

plt.tight_layout()
plt.show()

---

### 2.2 Normal Estimation via PCA

#### Mathematical Derivation

The surface normal at a point $\mathbf{p}$ is estimated by analysing the local geometry of
its $k$-nearest neighbourhood $\{\mathbf{p}_1, \ldots, \mathbf{p}_k\}$.

**Step 1 — Centroid:**

$$
\boldsymbol{\mu} = \frac{1}{k} \sum_{i=1}^{k} \mathbf{p}_i
$$

**Step 2 — Covariance matrix:**

$$
C = \frac{1}{k} \sum_{i=1}^{k} (\mathbf{p}_i - \boldsymbol{\mu})(\mathbf{p}_i - \boldsymbol{\mu})^T
\;\in\; \mathbb{R}^{3 \times 3}
$$

$C$ is symmetric positive semi-definite.  Its eigenvectors form an orthonormal basis
aligned with the principal axes of the local point distribution.

**Step 3 — Eigendecomposition via SVD:**

$$
C = U \Sigma U^T, \qquad \sigma_1 \geq \sigma_2 \geq \sigma_3 \geq 0.
$$

Since $C$ is symmetric, the SVD coincides with the eigendecomposition.  The eigenvalues
$\lambda_i = \sigma_i$ measure the variance of the point distribution along each
principal direction.

**Step 4 — Normal extraction:**

The eigenvector corresponding to the **smallest eigenvalue** $\lambda_3$ is the direction
of *least variance* — i.e., the surface normal:

$$
\hat{\mathbf{n}} = \mathbf{u}_3 \quad \text{(eigenvector for } \lambda_3\text{)}.
$$

**Planarity measure:**  $(\lambda_2 - \lambda_3) / \lambda_1$ — higher values indicate a
locally planar surface, which gives more reliable normal estimates.

**Orientation ambiguity:** PCA normals have a sign ambiguity ($\pm \hat{\mathbf{n}}$).
A common convention is to orient all normals towards the sensor (viewpoint), flipping
$\hat{\mathbf{n}}$ if $\hat{\mathbf{n}} \cdot (\mathbf{v} - \mathbf{p}) < 0$ where
$\mathbf{v}$ is the viewpoint.

#### From-Scratch Implementation

In [ ]:
def estimate_normals_scratch(points, k=20):
    """Estimate surface normals via PCA on local neighbourhoods (from scratch)."""
    tree = KDTree(points)
    _, idx = tree.query(points, k=k)                     # (N, k)

    neighbours = points[idx]                              # (N, k, 3)
    centroids = neighbours.mean(axis=1, keepdims=True)    # (N, 1, 3)
    diff = neighbours - centroids                         # (N, k, 3)

    cov = np.einsum('nki,nkj->nij', diff, diff) / k      # (N, 3, 3)

    _, _, Vt = np.linalg.svd(cov)                         # Vt: (N, 3, 3)
    normals = Vt[:, -1, :]                                # smallest eigenvalue

    # Orient normals towards the camera (assume camera at origin, +Z forward)
    flip = normals[:, 2] > 0
    normals[flip] *= -1

    norms = np.linalg.norm(normals, axis=1, keepdims=True).clip(min=1e-12)
    return normals / norms

In [ ]:
normals = estimate_normals_scratch(cloud_down, k=15)
print(f"Normals shape: {normals.shape}")
print(f"Mean norm: {np.linalg.norm(normals, axis=1).mean():.6f} (should be ~1.0)")

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

step = max(1, len(cloud_down) // 500)
sub_pts = cloud_down[::step]
sub_nrm = normals[::step]

ax.scatter(sub_pts[:, 0], sub_pts[:, 1], sub_pts[:, 2],
           c=sub_pts[:, 2], cmap='viridis', s=5, alpha=0.6)
ax.quiver(sub_pts[:, 0], sub_pts[:, 1], sub_pts[:, 2],
          sub_nrm[:, 0], sub_nrm[:, 1], sub_nrm[:, 2],
          length=0.05, color='red', alpha=0.6, linewidth=0.8)
ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
ax.set_title('Estimated Normals (quiver plot)')
ax.view_init(elev=25, azim=-60)
plt.tight_layout()
plt.show()

---

### 2.3 Statistical Outlier Removal

#### Mathematical Formulation

Sensor noise and multi-path reflections produce isolated points that lie far from the
true surface.  Statistical outlier removal identifies these by analysing the distribution
of nearest-neighbour distances.

For each point $\mathbf{p}_i$, compute the mean distance to its $k$ nearest neighbours:

$$
\bar{d}_i = \frac{1}{k} \sum_{j=1}^{k} \| \mathbf{p}_i - \mathbf{p}_{n_j} \|
$$

Then compute the global statistics of $\{\bar{d}_i\}$:

$$
\mu = \frac{1}{N} \sum_{i=1}^{N} \bar{d}_i, \qquad
\sigma = \sqrt{\frac{1}{N} \sum_{i=1}^{N} (\bar{d}_i - \mu)^2}.
$$

A point $\mathbf{p}_i$ is classified as an **inlier** if:

$$
\bar{d}_i \leq \mu + \alpha \cdot \sigma
$$

where $\alpha$ is a user-defined multiplier (typically $\alpha = 2$).  Under a Gaussian
assumption for $\{\bar{d}_i\}$, this retains approximately 95.4% of the distribution.

#### From-Scratch Implementation

In [ ]:
def remove_outliers_scratch(points, k=20, std_ratio=2.0):
    """Statistical outlier removal (from scratch)."""
    tree = KDTree(points[:, :3])
    dists, _ = tree.query(points[:, :3], k=k + 1)
    mean_dists = dists[:, 1:].mean(axis=1)  # skip self (distance 0)

    global_mean = mean_dists.mean()
    global_std = mean_dists.std()

    threshold = global_mean + std_ratio * global_std
    mask = mean_dists <= threshold
    return points[mask], mask

In [ ]:
np.random.seed(123)
n_outliers = 50
outlier_pts = np.random.uniform(
    low=cloud_down.min(axis=0) - 0.5,
    high=cloud_down.max(axis=0) + 0.5,
    size=(n_outliers, 3)
)
cloud_noisy = np.vstack([cloud_down, outlier_pts])

cloud_clean, inlier_mask = remove_outliers_scratch(cloud_noisy, k=15, std_ratio=2.0)

print(f"Before: {cloud_noisy.shape[0]:,} points")
print(f"After outlier removal: {cloud_clean.shape[0]:,} points")
print(f"Removed: {cloud_noisy.shape[0] - cloud_clean.shape[0]} points")

fig = plt.figure(figsize=(14, 5))
ax1 = fig.add_subplot(121, projection='3d')
colors = np.array(['steelblue'] * len(cloud_noisy))
colors[~inlier_mask] = 'red'
ax1.scatter(cloud_noisy[:, 0], cloud_noisy[:, 1], cloud_noisy[:, 2],
            c=colors, s=3, alpha=0.7)
ax1.set_title('With Outliers (red = outlier)')
ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')
ax1.view_init(elev=25, azim=-60)

ax2 = fig.add_subplot(122, projection='3d')
ax2.scatter(cloud_clean[:, 0], cloud_clean[:, 1], cloud_clean[:, 2],
            c=cloud_clean[:, 2], cmap='viridis', s=3, alpha=0.7)
ax2.set_title('After Outlier Removal')
ax2.set_xlabel('X'); ax2.set_ylabel('Y'); ax2.set_zlabel('Z')
ax2.view_init(elev=25, azim=-60)

plt.tight_layout()
plt.show()

---

## 3. Iterative Closest Point (ICP) Alignment

### 3.1 Problem Statement

Given two point clouds — a **source** $\mathcal{S} = \{\mathbf{s}_1, \ldots, \mathbf{s}_N\}$
and a **target** $\mathcal{T} = \{\mathbf{t}_1, \ldots, \mathbf{t}_M\}$ — that overlap
partially, find the rigid transformation $(R, \mathbf{t}) \in \text{SE}(3)$ that best
aligns $\mathcal{S}$ to $\mathcal{T}$:

$$
\min_{R \in SO(3),\; \mathbf{t} \in \mathbb{R}^3}
\sum_{i=1}^{N} \| R \mathbf{s}_i + \mathbf{t} - \mathbf{t}_{c(i)} \|^2
$$

where $c(i)$ maps each source point to its closest target point.

### 3.2 The ICP Algorithm (Besl & McKay, 1992)

ICP alternates between two steps:

1. **Correspondence step:** For each source point $\mathbf{s}_i$, find the closest target
   point $\mathbf{t}_{c(i)} = \arg\min_{\mathbf{t}_j \in \mathcal{T}} \| \mathbf{s}_i - \mathbf{t}_j \|$
   using a KD-Tree for $O(N \log M)$ complexity.

2. **Alignment step:** Given fixed correspondences, solve for the optimal $(R, \mathbf{t})$.

### 3.3 Closed-Form Solution via SVD (Arun et al., 1987)

Given $N$ pairs $(\mathbf{s}_i, \mathbf{t}_i)$, the optimal rigid transform minimises:

$$
E(R, \mathbf{t}) = \sum_{i=1}^{N} \| R \mathbf{s}_i + \mathbf{t} - \mathbf{t}_i \|^2.
$$

**Step 1 — Centre the points:**

$$
\bar{\mathbf{s}} = \frac{1}{N} \sum_{i=1}^{N} \mathbf{s}_i, \qquad
\bar{\mathbf{t}} = \frac{1}{N} \sum_{i=1}^{N} \mathbf{t}_i.
$$

Define centred coordinates $\mathbf{s}_i' = \mathbf{s}_i - \bar{\mathbf{s}}$ and
$\mathbf{t}_i' = \mathbf{t}_i - \bar{\mathbf{t}}$.  The optimal translation is:

$$
\mathbf{t}^* = \bar{\mathbf{t}} - R^* \bar{\mathbf{s}}.
$$

**Proof:** Setting $\partial E / \partial \mathbf{t} = 0$ yields
$\mathbf{t} = \bar{\mathbf{t}} - R \bar{\mathbf{s}}$.

**Step 2 — Cross-covariance matrix:**

Substituting $\mathbf{t}^*$ back, the problem reduces to maximising
$\text{trace}(R\, H)$ where:

$$
H = \sum_{i=1}^{N} \mathbf{s}_i' \, {\mathbf{t}_i'}^T \;\in\; \mathbb{R}^{3 \times 3}.
$$

**Step 3 — SVD of H:**

$$
H = U \Sigma V^T.
$$

The optimal rotation is:

$$
R^* = V U^T.
$$

**Reflection handling:** If $\det(V U^T) = -1$, the solution is a reflection, not a
rotation.  Fix by negating the column of $V$ corresponding to the smallest singular
value:

$$
R^* = V \, \text{diag}(1, 1, \det(VU^T)) \, U^T.
$$

**Step 4 — Optimal translation:**

$$
\mathbf{t}^* = \bar{\mathbf{t}} - R^* \bar{\mathbf{s}}.
$$

**Convergence:** ICP is guaranteed to monotonically decrease the error at each iteration.
However, it may converge to a **local minimum** — good initialisation is critical.

> **Proof sketch (monotonic convergence).** Let $E^{(k)} = E(R^{(k)}, \mathbf{t}^{(k)}, c^{(k)})$ denote the cost at iteration $k$.
>
> 1. **Correspondence step** (fix $(R, \mathbf{t})$, update $c$): Each new correspondence
>    $c^{(k+1)}(i) = \arg\min_j \|R^{(k)} \mathbf{s}_i + \mathbf{t}^{(k)} - \mathbf{t}_j\|$ minimises
>    its own summand, so:
>    $$E(R^{(k)}, \mathbf{t}^{(k)}, c^{(k+1)}) \leq E(R^{(k)}, \mathbf{t}^{(k)}, c^{(k)})$$
>
> 2. **Alignment step** (fix $c$, update $(R, \mathbf{t})$): The SVD solution is the
>    global minimum over all rigid transforms for the current correspondences:
>    $$E(R^{(k+1)}, \mathbf{t}^{(k+1)}, c^{(k+1)}) \leq E(R^{(k)}, \mathbf{t}^{(k)}, c^{(k+1)})$$
>
> Chaining gives $E^{(k+1)} \leq E^{(k)}$. Since $E \geq 0$ and the sequence is
> monotonically non-increasing, it converges by the monotone convergence theorem.
> The algorithm reaches a **local minimum** where no single nearest-neighbour
> reassignment reduces the cost — but this need not be the global minimum. $\square$

> **Why maximizing $\text{trace}(RH)$ gives $R = V U^\top$:** Let $H = U \Sigma V^\top$ be the SVD of the cross-covariance matrix. Then:
>
> $$\text{trace}(RH) = \text{trace}(R \cdot U \Sigma V^\top) = \text{trace}(\underbrace{V^\top R\, U}_{=: M} \cdot \Sigma)$$
>
> where we used the cyclic property $\text{trace}(ABC) = \text{trace}(CAB)$. Since $R$, $U$, $V$ are all orthogonal, $M = V^\top R\, U$ is also orthogonal. Now $\text{trace}(M \Sigma) = \sum_i m_{ii} \sigma_i \leq \sum_i \sigma_i$ because each diagonal entry of an orthogonal matrix satisfies $|m_{ii}| \leq 1$. Equality holds when $M = I$, i.e., $V^\top R\, U = I$, giving $R = V U^\top$.
>
> The $\text{diag}(1, 1, \det(VU^\top))$ correction ensures $\det(R) = +1$ (proper rotation) when the data is nearly coplanar and $VU^\top$ might yield a reflection.

### 3.4 Point-to-Point ICP — From-Scratch Implementation

In [ ]:
def icp_point_to_point(source, target, max_iter=50, tol=1e-6):
    """Point-to-point ICP alignment (from scratch).

    Returns: (T_4x4, per_point_distances, num_iterations, mse_history)
    """
    src = source[:, :3].copy()
    T_accum = np.eye(4, dtype=np.float64)
    tree = KDTree(target[:, :3])
    prev_mse = np.inf
    mse_history = []

    for iteration in range(max_iter):
        dists, idx = tree.query(src)
        matched = target[idx, :3]

        mse = float(np.mean(dists ** 2))
        mse_history.append(mse)

        src_mean = src.mean(axis=0)
        tgt_mean = matched.mean(axis=0)

        src_c = src - src_mean
        tgt_c = matched - tgt_mean

        H = src_c.T @ tgt_c
        U, S, Vt = np.linalg.svd(H)
        V = Vt.T

        d = np.linalg.det(V @ U.T)
        sign_matrix = np.diag([1.0, 1.0, d])
        R = V @ sign_matrix @ U.T
        t = tgt_mean - R @ src_mean

        src = (R @ src.T).T + t

        step = np.eye(4, dtype=np.float64)
        step[:3, :3] = R
        step[:3, 3] = t
        T_accum = step @ T_accum

        if abs(prev_mse - mse) < tol:
            dists_final, _ = tree.query(src)
            return T_accum, dists_final, iteration + 1, mse_history
        prev_mse = mse

    dists_final, _ = tree.query(src)
    return T_accum, dists_final, max_iter, mse_history

### 3.5 Point-to-Plane ICP Variant

Point-to-point ICP minimises distances to the closest points.  **Point-to-plane ICP**
(Chen & Medioni, 1992) instead minimises the distance along the target surface normal,
which converges faster when surfaces are locally planar.

**Convergence comparison** (Rusinkiewicz & Levoy, 3DIM 2001):

| Variant | Convergence rate | Normal estimation | Closed-form? |
|---------|:----------------:|:-----------------:|:------------:|
| Point-to-point | Linear | Not required | Yes (SVD) |
| **Point-to-plane** | **Superlinear** | Required | No (linearise) |
| Symmetric (Park et al., 2017) | Superlinear | Both clouds | No |
| Generalized ICP (Segal et al., 2009) | Quadratic | Covariance-based | No |

Point-to-plane converges faster because it allows points to "slide" along the
surface — only the *normal* component of the residual is penalised, not tangential
motion. This is why SLAM systems almost universally use point-to-plane ICP (or its
probabilistic generalisation, GICP).

#### Modified Objective

$$
E_{\text{plane}}(R, \mathbf{t})
= \sum_{i=1}^{N} \big[ (R \mathbf{s}_i + \mathbf{t} - \mathbf{t}_{c(i)}) \cdot \hat{\mathbf{n}}_{c(i)} \big]^2
$$

where $\hat{\mathbf{n}}_{c(i)}$ is the surface normal at the matched target point.

#### Linearised Solution

For small rotations, $R \approx I + [\boldsymbol{\omega}]_\times$ where
$\boldsymbol{\omega} = (\omega_x, \omega_y, \omega_z)^T$ and
$[\boldsymbol{\omega}]_\times$ is the skew-symmetric matrix.  Substituting:

$$
(R \mathbf{s}_i + \mathbf{t} - \mathbf{t}_{c(i)}) \cdot \hat{\mathbf{n}}_{c(i)}
\approx (\mathbf{s}_i + \boldsymbol{\omega} \times \mathbf{s}_i + \mathbf{t} - \mathbf{t}_{c(i)}) \cdot \hat{\mathbf{n}}_{c(i)} = 0.
$$

Using $\boldsymbol{\omega} \times \mathbf{s}_i = -\mathbf{s}_i \times \boldsymbol{\omega}$ and rearranging:

$$
\underbrace{\begin{pmatrix}
(\mathbf{s}_i \times \hat{\mathbf{n}}_i)^T & \hat{\mathbf{n}}_i^T
\end{pmatrix}}_{\mathbf{a}_i^T}
\underbrace{\begin{pmatrix} \boldsymbol{\omega} \\ \mathbf{t} \end{pmatrix}}_{\mathbf{x}}
= -(\mathbf{s}_i - \mathbf{t}_{c(i)}) \cdot \hat{\mathbf{n}}_i
$$

Stacking all $N$ equations: $A \mathbf{x} = \mathbf{b}$, solved by least squares
$(A^T A) \mathbf{x} = A^T \mathbf{b}$.

In [ ]:
def icp_point_to_plane(source, target, target_normals, max_iter=50, tol=1e-6):
    """Point-to-plane ICP with linearised least-squares (from scratch).

    Returns: (T_4x4, per_point_distances, num_iterations, mse_history)
    """
    src = source[:, :3].copy()
    T_accum = np.eye(4, dtype=np.float64)
    tree = KDTree(target[:, :3])
    prev_mse = np.inf
    mse_history = []

    for iteration in range(max_iter):
        dists, idx = tree.query(src)
        matched = target[idx, :3]
        matched_normals = target_normals[idx]

        mse = float(np.mean(dists ** 2))
        mse_history.append(mse)

        cross = np.cross(src, matched_normals)     # (N, 3)
        A = np.hstack([cross, matched_normals])    # (N, 6)
        b = -np.sum((src - matched) * matched_normals, axis=1)  # (N,)

        x, _, _, _ = np.linalg.lstsq(A, b, rcond=None)  # (6,)

        omega = x[:3]
        t_vec = x[3:]

        # Skew-symmetric → approximate rotation
        skew = np.array([[0, -omega[2], omega[1]],
                         [omega[2], 0, -omega[0]],
                         [-omega[1], omega[0], 0]])
        R = np.eye(3) + skew

        # Re-orthogonalise via SVD
        Ur, _, Vtr = np.linalg.svd(R)
        R = Ur @ Vtr
        if np.linalg.det(R) < 0:
            Ur[:, -1] *= -1
            R = Ur @ Vtr

        src = (R @ src.T).T + t_vec

        step = np.eye(4, dtype=np.float64)
        step[:3, :3] = R
        step[:3, 3] = t_vec
        T_accum = step @ T_accum

        if abs(prev_mse - mse) < tol:
            dists_final, _ = tree.query(src)
            return T_accum, dists_final, iteration + 1, mse_history
        prev_mse = mse

    dists_final, _ = tree.query(src)
    return T_accum, dists_final, max_iter, mse_history

### 3.6 ICP Exercise: Align Two Clouds

We generate a target cloud, apply a known rigid transform to create a source cloud,
then run both ICP variants to recover the transform.

In [ ]:
np.random.seed(7)
target_cloud = cloud_down.copy()

theta = np.radians(12)
R_true = np.array([[np.cos(theta), -np.sin(theta), 0],
                   [np.sin(theta),  np.cos(theta), 0],
                   [0,              0,             1]])
t_true = np.array([0.15, -0.08, 0.05])

source_cloud = (R_true @ target_cloud.T).T + t_true
source_cloud += np.random.normal(0, 0.002, source_cloud.shape)

print("Ground truth rotation (12° around Z):")
print(R_true)
print(f"Ground truth translation: {t_true}")

In [ ]:
T_p2p, dists_p2p, iters_p2p, mse_p2p = icp_point_to_point(
    source_cloud, target_cloud, max_iter=80, tol=1e-10
)
print(f"Point-to-point ICP converged in {iters_p2p} iterations")
print(f"Final MSE: {mse_p2p[-1]:.2e}")
print(f"Recovered rotation:\n{T_p2p[:3, :3]}")
print(f"Recovered translation: {T_p2p[:3, 3]}")

R_err = T_p2p[:3, :3] @ R_true
angle_err = np.degrees(np.arccos(np.clip((np.trace(R_err) - 1) / 2, -1, 1)))
t_err = np.linalg.norm(T_p2p[:3, 3] + R_true.T @ t_true)
print(f"\nRotation error: {angle_err:.4f}°")
print(f"Translation error: {t_err:.6f} m")

In [ ]:
target_normals_icp = estimate_normals_scratch(target_cloud, k=15)

T_p2l, dists_p2l, iters_p2l, mse_p2l = icp_point_to_plane(
    source_cloud, target_cloud, target_normals_icp, max_iter=80, tol=1e-10
)
print(f"Point-to-plane ICP converged in {iters_p2l} iterations")
print(f"Final MSE: {mse_p2l[-1]:.2e}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].semilogy(mse_p2p, 'b-o', markersize=3, label='Point-to-point')
axes[0].semilogy(mse_p2l, 'r-s', markersize=3, label='Point-to-plane')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('MSE (log scale)')
axes[0].set_title('ICP Convergence')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

src_aligned = pc.transform_points(source_cloud, T_p2p)
ax3d = fig.add_subplot(122, projection='3d')
sub = np.random.choice(len(target_cloud), min(2000, len(target_cloud)), replace=False)
ax3d.scatter(target_cloud[sub, 0], target_cloud[sub, 1], target_cloud[sub, 2],
             c='steelblue', s=2, alpha=0.5, label='Target')
ax3d.scatter(src_aligned[sub, 0], src_aligned[sub, 1], src_aligned[sub, 2],
             c='coral', s=2, alpha=0.5, label='Source (aligned)')
ax3d.set_title('After ICP Alignment')
ax3d.set_xlabel('X'); ax3d.set_ylabel('Y'); ax3d.set_zlabel('Z')
ax3d.legend()
ax3d.view_init(elev=25, azim=-60)

plt.tight_layout()
plt.show()

---

## 4. Poisson Surface Reconstruction

### 4.1 From Oriented Points to Watertight Meshes

Given a set of **oriented points** — 3-D positions with associated surface normals —
Poisson surface reconstruction (Kazhdan et al., 2006) produces a **watertight triangle
mesh** that approximates the underlying surface.

### 4.2 Mathematical Formulation

The key insight is that the surface normals define a **vector field** $\mathbf{V}(\mathbf{x})$
that should be the gradient of an **indicator function** $\Phi(\mathbf{x})$:

- $\Phi(\mathbf{x}) = 1$ inside the surface
- $\Phi(\mathbf{x}) = 0$ outside
- $\nabla \Phi \approx \mathbf{V}$ near the surface

We seek the scalar field $\Phi$ whose gradient best approximates $\mathbf{V}$ in the
least-squares sense:

$$
\Phi^* = \arg\min_\Phi \int_{\mathbb{R}^3} \|\nabla\Phi(\mathbf{x}) - \mathbf{V}(\mathbf{x})\|^2 \, d\mathbf{x}
$$

**Derivation via calculus of variations.** Let $\Phi = \Phi^* + \epsilon\,\eta$ for an
arbitrary test function $\eta$ with $\eta \to 0$ at infinity.  The first-order optimality
condition is:

$$
0 = \frac{d}{d\epsilon}\bigg|_{\epsilon=0}
\int \|\nabla(\Phi^* + \epsilon\,\eta) - \mathbf{V}\|^2 d\mathbf{x}
= 2\int \nabla\eta \cdot (\nabla\Phi^* - \mathbf{V})\, d\mathbf{x}
$$

Applying the divergence theorem (integration by parts in $\mathbb{R}^3$), the boundary
term vanishes because $\eta \to 0$ at infinity:

$$
0 = -2\int \eta\;\nabla \cdot (\nabla\Phi^* - \mathbf{V})\, d\mathbf{x}
$$

Since this holds for **all** test functions $\eta$, by the fundamental lemma of the
calculus of variations the integrand vanishes identically:

$$
\nabla \cdot \nabla\Phi^* = \nabla \cdot \mathbf{V}
$$

$$
\boxed{\Delta \Phi = \nabla \cdot \mathbf{V}}
$$

where $\Delta = \nabla^2 = \frac{\partial^2}{\partial x^2} + \frac{\partial^2}{\partial y^2} + \frac{\partial^2}{\partial z^2}$
is the **Laplacian operator**.  This is the **Poisson equation** — a classical second-order
elliptic PDE.  With Neumann or Dirichlet boundary conditions it has a unique solution
(up to an additive constant for Neumann), guaranteeing a well-defined indicator function
from which the iso-surface is extracted.

### 4.3 Computational Pipeline

The practical algorithm involves five major steps:

**1. Octree construction:** Build an adaptive octree over the point set.  Deeper levels
(more subdivisions) are used in regions with denser point samples, giving adaptive
resolution.

**2. B-spline basis functions:** Associate a smooth B-spline basis function $B_o(\mathbf{x})$
with each octree node $o$.  The indicator function is represented as:

$$
\Phi(\mathbf{x}) = \sum_{o} \alpha_o \, B_o(\mathbf{x})
$$

**3. Divergence computation:** Project the normal field $\mathbf{V}$ onto the basis:

$$
d_o = \langle \nabla \cdot \mathbf{V}, B_o \rangle
= -\langle \mathbf{V}, \nabla B_o \rangle
= -\sum_{p \in \text{samples}} \mathbf{n}_p \cdot \nabla B_o(\mathbf{p})
$$

**4. Sparse linear solve:** The discretised Laplacian gives a sparse linear system:

$$
L \boldsymbol{\alpha} = \mathbf{d}
$$

where $L_{ij} = \langle \Delta B_i, B_j \rangle$ is the Laplacian matrix.  This is solved
with a sparse conjugate gradient solver.

**5. Iso-surface extraction:** The surface is extracted as an iso-level of $\Phi$ using the
**Marching Cubes** algorithm.  The iso-value is typically chosen as the average of $\Phi$
evaluated at the input sample points.

### 4.4 Implementation Notes

A full from-scratch implementation of Poisson reconstruction would require:
- An octree data structure with neighbour-finding
- Tri-cubic B-spline evaluation and differentiation
- Sparse matrix assembly and solving
- Marching Cubes for iso-surface extraction

This is typically handled by libraries like **Open3D**.  Our workshop library wraps
Open3D's implementation in `src.pointcloud.poisson_reconstruct`.

In [ ]:
try:
    mesh = pc.poisson_reconstruct(cloud_down, normals, depth=6)
    vertices = np.asarray(mesh.vertices)
    triangles = np.asarray(mesh.triangles)
    print(f"Mesh: {len(vertices):,} vertices, {len(triangles):,} triangles")
except ImportError:
    print("Open3D not installed — skipping Poisson reconstruction.")
    print("Install with: pip install open3d")
    print("")
    print("Conceptual pipeline:")
    print("  1. Build octree over oriented points")
    print("  2. Define B-spline basis per octree node")
    print("  3. Compute divergence of normal field")
    print("  4. Solve Laplacian system: L @ alpha = d")
    print("  5. Extract iso-surface via Marching Cubes")

---

## 5. 3D Representation Comparison

Different 3-D tasks demand different representations.  Below we compare five major
representations used in modern 3-D vision and robotics.

### 5.1 Overview

| Representation | Storage | Query Time | Resolution | Adaptivity | Best For |
|:---|:---|:---|:---|:---|:---|
| **Point Cloud** | $O(N)$ | $O(\log N)$ KD-Tree | Sensor-limited | None (uniform) | Raw sensing, registration, SLAM |
| **Voxel Grid** | $O(n^3)$ | $O(1)$ index | Fixed grid | None | Occupancy mapping, collision checking, 3-D CNNs |
| **Octree** | $O(N_{\text{occ}})$ | $O(\log n)$ | Adaptive | High | Large-scale mapping, LoD rendering |
| **Triangle Mesh** | $O(V + F)$ | $O(\log F)$ BVH | Arbitrary | Manual | Rendering, CAD, physics simulation |
| **Neural Implicit** (NeRF/SDF) | $O(\theta)$ network | $O(1)$ forward pass | Continuous | Learned | Novel view synthesis, shape completion |

### 5.2 Detailed Discussion

**Point Cloud.** The simplest representation: an unordered set of $(x, y, z)$ coordinates,
possibly with per-point attributes (colour, normal, intensity).  Strengths: direct sensor
output, no discretisation, simple to merge from multiple viewpoints.  Weaknesses: no
surface connectivity, uneven density, inefficient for volume queries.

**Voxel Grid.** Discretises 3-D space into a regular grid.  Each voxel stores an occupancy
flag, TSDF value, or feature vector.  Strengths: $O(1)$ spatial queries, trivially
parallelisable, compatible with 3-D convolutions.  Weaknesses: cubic memory growth
$O(n^3)$, wastes storage on empty space, fixed resolution.

**Octree.** A hierarchical decomposition that recursively subdivides occupied voxels into
eight children.  Only allocates memory for non-empty regions, achieving $O(N_{\text{occ}})$
storage.  Supports level-of-detail (LoD) rendering.  Used in Poisson reconstruction,
OctoMap (Hornung et al., 2013).

**Triangle Mesh.** Vertices connected by triangular faces.  The standard representation
for rendering (rasterisation) and physics simulation.  Supports texture mapping, smooth
shading via vertex normals, and efficient ray intersection via BVH trees.  Generating
high-quality meshes from point clouds requires algorithms like Poisson reconstruction
or Ball Pivoting.

**Neural Implicit (NeRF / SDF).** Represents geometry as a continuous function
$f_\theta(\mathbf{x}) \to \text{value}$ parameterised by a neural network:

- **NeRF** (Mildenhall et al., 2020): $f_\theta(\mathbf{x}, \mathbf{d}) \to (\mathbf{c}, \sigma)$
  maps position + view direction to colour + density.
- **DeepSDF** (Park et al., 2019): $f_\theta(\mathbf{x}) \to s$ maps position to signed
  distance value.  The surface is the zero-level set $\{\mathbf{x} : f_\theta(\mathbf{x}) = 0\}$.

Strengths: continuous resolution, compact storage ($O(|\theta|)$), differentiable.  
Weaknesses: slow inference (many forward passes for rendering), training required,
difficult to edit or compose. For autonomous drones, neural implicits enable compact scene representations that can be queried for collision checking at arbitrary resolution, but their inference latency currently limits them to offline planning rather than real-time obstacle avoidance.

### 5.3 Occupancy Networks — Decision Boundary (Mescheder et al., 2019)

An occupancy network predicts the probability that a query point $\mathbf{x}$ lies
**inside** an object:

$$
f_\theta(\mathbf{x}, \mathbf{z}) = \sigma\!\bigl(g_\theta(\mathbf{x}, \mathbf{z})\bigr) \in [0, 1]
$$

where $\mathbf{z}$ is a latent shape code, $g_\theta$ is the network's logit output,
and $\sigma(t) = 1/(1 + e^{-t})$ is the sigmoid.  The **decision boundary** (surface)
is the 0.5-level set:

$$
\mathcal{S} = \bigl\{\mathbf{x} : f_\theta(\mathbf{x}, \mathbf{z}) = 0.5\bigr\}
= \bigl\{\mathbf{x} : g_\theta(\mathbf{x}, \mathbf{z}) = 0\bigr\}
$$

since $\sigma(0) = 0.5$.

**Training loss (binary cross-entropy).** Given $N$ sampled query points $\mathbf{x}_i$
with ground-truth occupancy labels $o_i \in \{0, 1\}$:

$$
\mathcal{L}_{\text{BCE}} = -\frac{1}{N}\sum_{i=1}^{N}\bigl[o_i \log f_\theta(\mathbf{x}_i)
+ (1 - o_i)\log\bigl(1 - f_\theta(\mathbf{x}_i)\bigr)\bigr]
$$

**Derivation.** Each query point is modelled as a Bernoulli variable
$o_i \sim \text{Bern}(f_\theta(\mathbf{x}_i))$.  The negative log-likelihood of the
observed labels is exactly the BCE above, so minimising $\mathcal{L}_{\text{BCE}}$
is maximum-likelihood estimation of the occupancy field.

**Occupancy vs. SDF at the decision boundary:**

| Property | Occupancy network | DeepSDF |
|:---|:---|:---|
| Output | $[0,1]$ (probability) | $\mathbb{R}$ (signed distance) |
| Surface condition | $f = 0.5$ ($g = 0$) | $f = 0$ |
| Loss | Binary cross-entropy | L1 regression + Eikonal |
| Gradient at surface | Steep sigmoid transition | $\|\nabla f\| = 1$ (Eikonal) |

The **Eikonal regulariser** $\mathcal{L}_{\text{eik}} = \mathbb{E}_{\mathbf{x}}\bigl[(\|\nabla_{\mathbf{x}} f_\theta\| - 1)^2\bigr]$
enforces the SDF property that the gradient magnitude equals 1 everywhere,
producing smoother and more physically meaningful distance fields.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Point cloud
ax = fig.add_subplot(131, projection='3d')
sub = np.random.choice(len(cloud_down), min(2000, len(cloud_down)), replace=False)
ax.scatter(cloud_down[sub, 0], cloud_down[sub, 1], cloud_down[sub, 2],
           c=cloud_down[sub, 2], cmap='viridis', s=2)
ax.set_title('Point Cloud')
ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
ax.view_init(elev=25, azim=-60)

# 2. Voxel grid
voxel_size_vis = 0.12
voxel_centres = voxel_downsample_scratch(cloud_down, voxel_size_vis)
ax = fig.add_subplot(132, projection='3d')
ax.scatter(voxel_centres[:, 0], voxel_centres[:, 1], voxel_centres[:, 2],
           c=voxel_centres[:, 2], cmap='viridis', s=40, marker='s', alpha=0.7)
ax.set_title(f'Voxel Grid (s={voxel_size_vis})')
ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
ax.view_init(elev=25, azim=-60)

# 3. Normals (proxy for mesh information)
ax = fig.add_subplot(133, projection='3d')
step = max(1, len(cloud_down) // 400)
sp = cloud_down[::step]
sn = normals[::step]
ax.scatter(sp[:, 0], sp[:, 1], sp[:, 2], c=sp[:, 2], cmap='viridis', s=5)
ax.quiver(sp[:, 0], sp[:, 1], sp[:, 2],
          sn[:, 0], sn[:, 1], sn[:, 2],
          length=0.04, color='red', alpha=0.5)
ax.set_title('Oriented Points (for meshing)')
ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')

plt.tight_layout()
plt.show()

---

## 6. SE(3)-Equivariant Networks

**SE(3)-equivariant** networks guarantee that if the input is rigidly transformed, the output transforms correspondingly — by construction, not by data augmentation:

$$
f(g \cdot \mathbf{x}) = \rho(g) \cdot f(\mathbf{x}), \qquad \forall g \in \text{SE}(3)
$$

**How it works**: convolution kernels are expanded in a basis of spherical harmonics $Y_\ell^m$ (where $\ell \in \{0, 1, 2, \ldots\}$ is the angular frequency (order) of the spherical harmonic) and learnable radial functions $R_\ell$:

$$
\kappa(\mathbf{r}) = \sum_{\ell, m} R_\ell(\|\mathbf{r}\|) \, Y_\ell^m(\hat{\mathbf{r}})
$$

Features at each point are decomposed into irreducible representations of SO(3) (scalars $\ell=0$, vectors $\ell=1$, matrices $\ell=2$, ...). Rotating the input rotates features via Wigner D-matrices.

**Simplest variant — EGNN** (Satorras et al., 2021): achieves E(n)-equivariance with just coordinate updates:

$$
\mathbf{x}_i' = \mathbf{x}_i + \frac{1}{|\mathcal{N}(i)|}
\sum_{j \in \mathcal{N}(i)} (\mathbf{x}_i - \mathbf{x}_j) \, \phi(\mathbf{h}_i, \mathbf{h}_j, \|\mathbf{x}_i - \mathbf{x}_j\|^2)
$$

**Relevance to 3D vision**: SE(3)-equivariant features enable rotation-robust point cloud registration and object detection without augmentation. See Thomas et al. (2018), Fuchs et al. (2020), Brandstetter et al. (2022).

## 7. Perspective-n-Point (PnP)

Given $n$ known 3D-2D correspondences $(\mathbf{X}_i, \mathbf{x}_i)$, solve
for the camera pose $[R \mid \mathbf{t}]$.

### Why PnP Is Here
PnP is essential for:
- **SfM image registration** (NB 10): register new images using known 3D points
- **SLAM relocalization** (NB 14): recover pose after tracking loss
- **Stereo VO** (NB 07): estimate pose from 3D-2D matches

### Historical Context and Solver Taxonomy

The PnP problem has a rich history stretching back to 1841 (Grunert). Modern
solvers balance speed, accuracy, and numerical stability:

| Solver | Points | Complexity | Properties | Reference |
|--------|:------:|-----------|-----------|-----------|
| **P3P** | 3 (minimal) | $O(1)$ | Up to 4 solutions; use inside RANSAC | Grunert 1841; Fischler & Bolles 1981 |
| **DLT** | $\geq 6$ | $O(n)$ | Simple SVD; no intrinsics separation | Abdel-Aziz & Karara 1971 |
| **EPnP** | $\geq 4$ | $O(n)$ | Express 3D points as barycentric coords of 4 control points | Lepetit et al., IJCV 2009 |
| **SQPnP** | $\geq 3$ | $O(n)$ | SDP relaxation + sequential QP refinement | Terzakis & Lourakis, ECCV 2020 |
| **P3P + RANSAC** | $\geq 3$ | $O(n \cdot k)$ | Robust to outliers; gold standard for SLAM | — |

**EPnP** is the most widely used non-minimal solver: it reduces PnP to finding
the positions of 4 *virtual control points* that define the reference frame.
The 3D-2D constraints become linear in the control-point coordinates, solved via
SVD of a $2n \times 12$ matrix. This avoids the numerically unstable quartic
polynomial of P3P and runs in $O(n)$ time.

### P3P: The Minimal Solver

With 3 correspondences, the problem reduces to finding distances along 3
back-projected rays. Using the **law of cosines** on triangles formed by the
3D points:

$$
\|\mathbf{X}_i - \mathbf{X}_j\|^2 = d_i^2 + d_j^2 - 2 d_i d_j \cos\theta_{ij}
$$

where $d_i$ is the unknown distance along ray $i$ and $\theta_{ij}$ is the
angle between rays (known from camera intrinsics). This gives a system of 3
equations in 3 unknowns, yielding **up to 4 solutions**.

### PnP + RANSAC

In practice, use RANSAC:
1. Sample 3 correspondences
2. Solve P3P → up to 4 solutions
3. For each solution, count inliers (reprojection error < threshold)
4. Keep best, refine with all inliers via Levenberg-Marquardt

OpenCV: `cv2.solvePnPRansac(objectPoints, imagePoints, K, dist)`

> **PnP DLT: two equations per correspondence.** Given a 3D point $\mathbf{X} = (X, Y, Z, 1)^\top$ and its 2D projection $\mathbf{x} = (u, v, 1)^\top$, the projection $\mathbf{x} \sim P\mathbf{X}$ (where $P$ is the $3 \times 4$ camera matrix) means:
>
> $$u = \frac{\mathbf{p}_1^\top \mathbf{X}}{\mathbf{p}_3^\top \mathbf{X}}, \quad v = \frac{\mathbf{p}_2^\top \mathbf{X}}{\mathbf{p}_3^\top \mathbf{X}}$$
>
> Cross-multiplying:
>
> $$u \cdot \mathbf{p}_3^\top \mathbf{X} - \mathbf{p}_1^\top \mathbf{X} = 0, \quad v \cdot \mathbf{p}_3^\top \mathbf{X} - \mathbf{p}_2^\top \mathbf{X} = 0$$
>
> Each correspondence gives 2 linear equations in the 12 entries of $P$. With $\geq 6$ correspondences, we get the overdetermined system $A\mathbf{p} = 0$ (where $\mathbf{p} = \text{vec}(P)$), solved via SVD.

In [ ]:
from src.transforms import rotation_matrix_from_euler, se3_from_Rt

R_gt = rotation_matrix_from_euler(0.2, -0.1, 0.3)
t_gt = np.array([1.0, -0.5, 3.0])

K_pnp = np.array([[500, 0, 320], [0, 500, 240], [0, 0, 1]], dtype=np.float64)

n_pts = 20
world_pts = np.random.uniform(-2, 2, (n_pts, 3))
world_pts[:, 2] += 5  # ensure in front of camera

cam_pts = (R_gt @ world_pts.T).T + t_gt
proj = K_pnp @ cam_pts.T
pixels = (proj[:2] / proj[2:]).T

pixels_noisy = pixels + np.random.normal(0, 1.0, pixels.shape)

def solve_pnp_dlt(world, pixels, K):
    """Direct Linear Transform for PnP (6+ points)."""
    K_inv = np.linalg.inv(K)
    ones = np.ones((len(pixels), 1))
    norm_pts = (K_inv @ np.hstack([pixels, ones]).T).T[:, :2]
    
    A = []
    for (X, Y, Z), (u, v) in zip(world, norm_pts):
        A.append([X, Y, Z, 1, 0, 0, 0, 0, -u*X, -u*Y, -u*Z, -u])
        A.append([0, 0, 0, 0, X, Y, Z, 1, -v*X, -v*Y, -v*Z, -v])
    A = np.array(A)
    
    _, _, Vt = np.linalg.svd(A)
    P = Vt[-1].reshape(3, 4)
    
    M = P[:, :3]
    U, S, Vt2 = np.linalg.svd(M)
    R = U @ Vt2
    scale = np.mean(S)
    if np.linalg.det(R) < 0:
        R = -R
        scale = -scale
    t = P[:, 3] / scale
    
    return R, t

R_est, t_est = solve_pnp_dlt(world_pts, pixels_noisy, K_pnp)

rot_error = np.degrees(np.arccos(np.clip((np.trace(R_gt.T @ R_est) - 1) / 2, -1, 1)))
trans_error = np.linalg.norm(t_gt - t_est)

print(f"PnP Results (DLT, {n_pts} points, 1px noise):")
print(f"  Rotation error:    {rot_error:.2f}°")
print(f"  Translation error: {trans_error:.4f} m")
print(f"\nGT translation: {t_gt}")
print(f"Est translation: {t_est}")

---

## 8. Exercises

### Exercise 1: Generate a Point Cloud from a Synthetic Depth Map (Sphere + Plane)

Create a depth map containing a sphere floating above a ground plane.  Back-project it
to a 3-D point cloud and visualise.

In [ ]:
H_ex, W_ex = 100, 140
fx_ex, fy_ex = 250.0, 250.0
cx_ex, cy_ex = W_ex / 2.0, H_ex / 2.0

u_ex, v_ex = np.meshgrid(np.arange(W_ex), np.arange(H_ex))

depth_ex = 5.0 * np.ones((H_ex, W_ex))

sc_u, sc_v = 70, 40
sphere_r_px = 20
sphere_d0 = 3.5
r_dist = np.sqrt((u_ex - sc_u)**2 + (v_ex - sc_v)**2)
sphere_mask_ex = r_dist < sphere_r_px
sphere_bump = np.sqrt(np.maximum(sphere_r_px**2 - r_dist**2, 0)) / fx_ex
depth_ex[sphere_mask_ex] = sphere_d0 - sphere_bump[sphere_mask_ex]

pts_ex = depth_to_pointcloud_scratch(depth_ex, fx_ex, fy_ex, cx_ex, cy_ex)
print(f"Exercise 1 point cloud: {pts_ex.shape[0]:,} points")

fig = plt.figure(figsize=(12, 5))
ax1 = fig.add_subplot(121)
ax1.imshow(depth_ex, cmap='viridis')
ax1.set_title('Depth Map (Sphere + Plane)')

ax2 = fig.add_subplot(122, projection='3d')
sub = np.random.choice(len(pts_ex), min(5000, len(pts_ex)), replace=False)
ax2.scatter(pts_ex[sub, 0], pts_ex[sub, 1], pts_ex[sub, 2],
            c=pts_ex[sub, 2], cmap='viridis', s=1)
ax2.set_xlabel('X'); ax2.set_ylabel('Y'); ax2.set_zlabel('Z')
ax2.set_title('Back-Projected Point Cloud')
plt.tight_layout()
plt.show()

### Exercise 2: Voxel Downsample and Compare Point Counts

Downsample the Exercise 1 cloud at three different voxel sizes and compare the resulting
point counts.

In [ ]:
voxel_sizes = [0.02, 0.05, 0.10]
print(f"{'Voxel size':>12s} | {'Points':>8s} | {'Ratio':>8s}")
print('-' * 35)
print(f"{'Original':>12s} | {pts_ex.shape[0]:>8,} | {'100.0%':>8s}")

fig = plt.figure(figsize=(16, 4))
for i, vs in enumerate(voxel_sizes):
    ds = voxel_downsample_scratch(pts_ex, vs)
    ratio = ds.shape[0] / pts_ex.shape[0]
    print(f"{vs:>12.2f} | {ds.shape[0]:>8,} | {ratio:>7.1%}")

    ax = fig.add_subplot(1, 3, i + 1, projection='3d')
    ax.scatter(ds[:, 0], ds[:, 1], ds[:, 2],
              c=ds[:, 2], cmap='viridis', s=max(1, 8 * vs / 0.02))
    ax.set_title(f's = {vs}')
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')

plt.tight_layout()
plt.show()

### Exercise 3: Estimate Normals and Visualise with Quiver Plot

Estimate normals on the downsampled Exercise 1 cloud and display a 3-D quiver plot.

In [ ]:
cloud_ex_ds = voxel_downsample_scratch(pts_ex, 0.04)
normals_ex = estimate_normals_scratch(cloud_ex_ds, k=20)

print(f"Downsampled points: {cloud_ex_ds.shape[0]:,}")
print(f"Normal norms — mean: {np.linalg.norm(normals_ex, axis=1).mean():.4f}, "
      f"std: {np.linalg.norm(normals_ex, axis=1).std():.4f}")

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

step = max(1, len(cloud_ex_ds) // 600)
sp = cloud_ex_ds[::step]
sn = normals_ex[::step]

ax.scatter(sp[:, 0], sp[:, 1], sp[:, 2], c=sp[:, 2], cmap='viridis', s=5, alpha=0.6)
ax.quiver(sp[:, 0], sp[:, 1], sp[:, 2],
          sn[:, 0], sn[:, 1], sn[:, 2],
          length=0.06, color='crimson', alpha=0.5, linewidth=0.8)
ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
ax.set_title('Exercise 3: Normals on Sphere + Plane')
plt.tight_layout()
plt.show()

### Exercise 4: Align Two Clouds with ICP and Plot Convergence

Create a rotated+translated copy of the Exercise 1 cloud, then align with ICP.  Plot the
MSE convergence curve.

In [ ]:
np.random.seed(99)

angle_ex = np.radians(8)
R_ex = np.array([[1, 0, 0],
                 [0, np.cos(angle_ex), -np.sin(angle_ex)],
                 [0, np.sin(angle_ex),  np.cos(angle_ex)]])
t_ex = np.array([0.1, -0.05, 0.03])

target_ex = cloud_ex_ds.copy()
source_ex = (R_ex @ cloud_ex_ds.T).T + t_ex
source_ex += np.random.normal(0, 0.001, source_ex.shape)

T_result, _, iters_ex, mse_ex = icp_point_to_point(
    source_ex, target_ex, max_iter=100, tol=1e-12
)

print(f"Converged in {iters_ex} iterations")
print(f"Initial MSE: {mse_ex[0]:.6f}")
print(f"Final MSE:   {mse_ex[-1]:.2e}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].semilogy(mse_ex, 'b-o', markersize=3)
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('MSE (log scale)')
axes[0].set_title('Exercise 4: ICP Convergence')
axes[0].grid(True, alpha=0.3)

aligned_ex = pc.transform_points(source_ex, T_result)
ax3d = fig.add_subplot(122, projection='3d')
sub = np.random.choice(len(target_ex), min(2000, len(target_ex)), replace=False)
ax3d.scatter(target_ex[sub, 0], target_ex[sub, 1], target_ex[sub, 2],
             c='steelblue', s=2, alpha=0.5, label='Target')
ax3d.scatter(aligned_ex[sub, 0], aligned_ex[sub, 1], aligned_ex[sub, 2],
             c='coral', s=2, alpha=0.5, label='Aligned source')
ax3d.set_title('After ICP')
ax3d.legend()
ax3d.set_xlabel('X'); ax3d.set_ylabel('Y'); ax3d.set_zlabel('Z')

plt.tight_layout()
plt.show()

### Exercise 5 (Bonus): Surface Reconstruction Concept

Given oriented points, sketch the Poisson reconstruction pipeline.  If Open3D is
available, run it; otherwise, describe the steps and verify that the library function
has the correct interface.

In [ ]:
print("Poisson Surface Reconstruction — Conceptual Pipeline")
print("=" * 55)
print()
print("Input:  Oriented point cloud (N points with normals)")
print(f"        Our cloud: {cloud_ex_ds.shape[0]} points, {normals_ex.shape[0]} normals")
print()
print("Step 1: Build adaptive octree (depth d controls resolution)")
print("        Each level doubles spatial resolution: 2^d cells per axis")
print("        depth=8 → 256³ = 16.7M potential cells (but octree is sparse)")
print()
print("Step 2: Associate B-spline basis B_o(x) with each octree node o")
print("        Indicator function: Φ(x) = Σ α_o · B_o(x)")
print()
print("Step 3: Compute divergence d_o = -Σ_p n_p · ∇B_o(p)")
print("        This projects the normal field onto the basis")
print()
print("Step 4: Solve sparse system L·α = d  (Poisson equation)")
print("        L_ij = <ΔB_i, B_j>  (Laplacian matrix)")
print("        Solved with conjugate gradient")
print()
print("Step 5: Extract iso-surface of Φ via Marching Cubes")
print("        iso-value = mean of Φ at input sample locations")
print()
print("Output: Watertight triangle mesh (vertices + faces)")
print()

try:
    mesh_ex = pc.poisson_reconstruct(cloud_ex_ds, normals_ex, depth=6)
    verts = np.asarray(mesh_ex.vertices)
    tris = np.asarray(mesh_ex.triangles)
    print(f"\nResult: {len(verts):,} vertices, {len(tris):,} triangles")
except ImportError:
    print("(Open3D not installed — conceptual exercise complete)")
except Exception as e:
    print(f"(Reconstruction failed: {e} — this is expected for noisy synthetic data)")

---

## Limitations & Failure Cases

- **ICP local minima:** ICP is guaranteed to decrease the cost at each iteration, but it converges to the nearest local minimum — not necessarily the global one. Without a good initial alignment (e.g. starting with >45° rotation error), ICP often converges to a completely wrong pose.
- **Point cloud noise:** Noisy depth measurements produce noisy point clouds, which in turn yield noisy surface normals from PCA. This propagates into poor surface reconstruction and unreliable ICP convergence.
- **PnP degenerate cases:** PnP fails when 3D–2D correspondences are coplanar (the solution becomes ambiguous), when fewer than 4 correspondences are available, or when the outlier ratio is high (requiring RANSAC with many iterations).
- **Voxel resolution trade-off:** Voxel grids face a fundamental resolution–memory trade-off: too coarse and fine geometric detail is lost; too fine and memory usage grows as $O(1/s^3)$ where $s$ is the voxel size. Adaptive structures (octrees) mitigate this but add complexity.
- **Poisson reconstruction artifacts:** Poisson surface reconstruction requires consistently oriented normals. When normal orientations are inconsistent (common at depth discontinuities or thin structures), the reconstructed mesh develops holes, bubbles, or phantom surfaces.

---

## Summary

In this notebook we covered the full pipeline from raw depth measurements to processed
3-D representations:

| Topic | Key Equation | Section |
|:---|:---|:---|
| Back-projection | $X = (u - c_x) d / f_x$ | §1 |
| Voxel downsample | $\mathbf{v} = \lfloor \mathbf{p}/s \rfloor$ | §2.1 |
| Normal estimation | $C = \frac{1}{k} \sum (\mathbf{p}_i - \boldsymbol{\mu})(\mathbf{p}_i - \boldsymbol{\mu})^T$ | §2.2 |
| Outlier removal | $\bar{d}_i \leq \mu + \alpha \sigma$ | §2.3 |
| ICP (point-to-point) | $R^* = V U^T$, $\mathbf{t}^* = \bar{\mathbf{t}} - R^* \bar{\mathbf{s}}$ | §3 |
| ICP (point-to-plane) | Linearised: $A\mathbf{x} = \mathbf{b}$ | §3.5 |
| Poisson reconstruction | $\Delta \Phi = \nabla \cdot \mathbf{V}$ | §4 |

### Key Takeaways

1. **Back-projection** is the inverse of the pinhole projection — every pixel + depth gives a 3-D point.
2. **Voxel downsampling** is the workhorse of point cloud simplification — $O(N \log N)$, trivially parallel.
3. **PCA normals** via SVD of local covariance matrices — the smallest eigenvector is the normal.
4. **ICP** alternates NN correspondences with SVD-based alignment — guaranteed monotonic descent but may find local minima.
5. **Point-to-plane ICP** converges faster by exploiting surface orientation.
6. **Poisson reconstruction** converts oriented points to watertight meshes by solving the Laplacian PDE.
7. The choice of 3-D representation (points / voxels / meshes / neural) depends on the downstream task.

**Next notebook: Notebook 12 — Volumetric Mapping (TSDF, Occupancy Grids)**